# Report 4 — XGBoost: huấn luyện và tinh chỉnh (phat)

Phần việc: tinh chỉnh tham số XGBoost bằng CV theo kênh, chọn ngưỡng theo F1, giao bộ tham số đã chốt cho duc.

**Quy ước** (`docs/TEAM_HANDOVER_REPORT4.md`): tập "có khả năng trending" (bỏ Shorts, Tin tức và Chính trị, Thể thao); bộ feature C cho cây; tập test **không chấm ở đây**; chọn tham số theo PR-AUC trung bình; `scale_pos_weight` tính trong từng fold; `random_state=42`.

Cần `uv sync --group xgboost` và file `data/processed/video_features_full_2026-11-01.csv`.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

ROOT = Path.cwd()
if not (ROOT / "src").exists():          # chạy từ notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.modeling import model as M

CSV = ROOT / "data" / "processed" / "video_features_full_2026-11-01.csv"
TEST_CSV = ROOT / "data" / "processed" / "test_channels.csv"
PARAMS_JSON = ROOT / "data" / "processed" / "xgb_best_params.json"

## 1. Dữ liệu và tách train/test

Dùng đúng danh sách kênh test đã commit (`test_channels.csv`), không tạo lại. Tập chính = `is_clean`, bỏ Shorts, Tin tức và Chính trị, Thể thao (giống `loc_co_kha_nang()` trong `split.py`).
Nếu bạn muốn dùng thẳng `doc_tap_sach()` / `tach_train_test()` của `split.py`, thay ô này nhưng giữ nguyên các dòng `assert`.

In [ ]:
raw = pd.read_csv(CSV, low_memory=False)
clean = raw[M._to_bool(raw["is_clean"])].reset_index(drop=True)
clean["y"] = M._to_bool(clean[M.LABEL])

if TEST_CSV.exists():
    test_channels = set(pd.read_csv(TEST_CSV)[M.GROUP])
else:   # chỉ khi chưa pull test_channels.csv: tái tạo đúng cách của BanNH (kết quả giống hệt)
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    _, te = next(sgkf.split(clean, clean["y"], groups=clean[M.GROUP]))
    test_channels = set(clean[M.GROUP].iloc[te])

co_kha_nang = (~M._to_bool(clean["is_short"])) & (~clean["category_title"].isin(["Tin tức và Chính trị", "Thể thao"]))
main = clean[co_kha_nang].reset_index(drop=True)
is_test = main[M.GROUP].isin(test_channels)
train = main[~is_test].reset_index(drop=True)

# Số liệu phải khớp mục 5.1 của handover. Không đụng tới nhãn của tập test.
assert len(train) == 8152 and train[M.GROUP].nunique() == 1100 and int(train["y"].sum()) == 315, "train không khớp handover"
assert int(is_test.sum()) == 2079, "test không khớp handover"
print(f"train: {len(train)} video, {train[M.GROUP].nunique()} kênh, {int(train['y'].sum())} trending ({train['y'].mean():.2%})")

train: 8152 video, 1100 kênh, 315 trending (3.86%)


## 2. Feature (bộ C, mô hình cây) và kiểm tra fold

In [ ]:
X, y, groups, cols = M.build_features(train, "C", model_type="tree")
print(X.shape); print(cols)
display(X.head(3))
display(M.check_folds(y, groups))          # không kênh nào ở cả train và validation
print("scale_pos_weight toàn train:", round(M.scale_pos_weight(y), 1), "(trong CV tính lại theo từng fold)")

(8152, 12)
{'num': ['publish_hour_vn', 'publish_dow_vn', 'is_weekend', 'log10_subs', 'is_live_or_premiere', 'title_length', 'n_tags', 'log10_duration'], 'cat': ['time_slot', 'slot_detail', 'release_date', 'category_group']}


,time_slot,slot_detail,publish_hour_vn,publish_dow_vn,is_weekend,release_date,category_group,log10_subs,is_live_or_premiere,title_length,n_tags,log10_duration
0,hanh_chinh,gio_hanh_chinh,16,1,0,2026-09-28,Âm nhạc,4.704159,0,100,21,3.703119
1,hanh_chinh,gio_hanh_chinh,11,5,0,2026-10-02,Âm nhạc,3.645521,0,98,30,3.911424
2,hanh_chinh,gio_hanh_chinh,10,3,0,2026-09-30,Âm nhạc,4.260095,0,76,23,2.571709


,fold,n_train,n_val,pos_train,pos_val
0,0,6521,1631,252,63
1,1,6523,1629,253,62
2,2,6524,1628,252,63
3,3,6520,1632,252,63
4,4,6520,1632,251,64


scale_pos_weight toàn train: 24.9 (trong CV tính lại theo từng fold)


## 3. Tinh chỉnh tham số

Lưới `M.XGB_GRID` = 3 × 2 × 2 × 2 × 2 = 48 tổ hợp, mỗi tổ hợp 5 fold theo kênh. Chọn theo **PR-AUC trung bình**. Tiền xử lý (one-hot) và `scale_pos_weight` đều fit trong từng fold.
Nếu chạy quá lâu, thu nhỏ lưới bằng cách truyền `grid=` cho `tune_xgb`.

In [ ]:
res = M.tune_xgb(X, y, groups, cols, verbose=False)
display(res["table"].head(10).round(4))

,max_depth,learning_rate,n_estimators,min_child_weight,subsample,pr_auc_mean,pr_auc_std
0,4,0.03,200,5,0.8,0.2398,0.0826
1,3,0.03,400,5,1.0,0.2398,0.0829
2,3,0.03,200,5,0.8,0.2352,0.0859
3,3,0.03,400,5,0.8,0.2340,0.0840
4,3,0.03,400,1,1.0,0.2321,0.0838
5,3,0.03,200,5,1.0,0.2316,0.0858
6,4,0.03,400,5,0.8,0.2313,0.0770
7,4,0.03,200,1,0.8,0.2310,0.0715
8,3,0.03,400,1,0.8,0.2305,0.0799
9,3,0.10,200,5,0.8,0.2304,0.0811


## 4. Bộ tham số đã chốt và ngưỡng

In [ ]:
best = res["best_params"]; thr = res["threshold"]
print("tham số:", best)
print(f"PR-AUC CV: {res['best_pr_auc_mean']:.4f} ± {res['best_pr_auc_std']:.4f}  (tỉ lệ trending = {y.mean():.4f})")
print(f"ngưỡng F1 chọn trong CV: {thr:.4f}")

tbl = res["table"]
n_close = int((tbl["pr_auc_mean"] >= res["best_pr_auc_mean"] - res["best_pr_auc_std"]).sum())
print(f"{n_close}/{len(tbl)} tổ hợp nằm trong 1 độ lệch chuẩn của tổ hợp tốt nhất "
      "-> nếu số này lớn, việc chọn tham số không phân biệt rõ, đừng diễn giải quá mức.")

PARAMS_JSON.write_text(json.dumps({
    "feature_set": "C", "model_type": "tree", "random_state": M.RANDOM_STATE,
    "best_params": best, "threshold": thr,
    "pr_auc_cv_mean": res["best_pr_auc_mean"], "pr_auc_cv_std": res["best_pr_auc_std"],
    "scale_pos_weight_note": "tính lại trong từng fold: số nhãn 0 / số nhãn 1",
}, indent=2, ensure_ascii=False), encoding="utf-8")
print("đã ghi", PARAMS_JSON)

tham số: {'max_depth': 4, 'learning_rate': 0.03, 'n_estimators': 200, 'min_child_weight': 5, 'subsample': 0.8}
PR-AUC CV: 0.2398 ± 0.0826  (tỉ lệ trending = 0.0386)
ngưỡng F1 chọn trong CV: 0.7774
48/48 tổ hợp nằm trong 1 độ lệch chuẩn của tổ hợp tốt nhất -> nếu số này lớn, việc chọn tham số không phân biệt rõ, đừng diễn giải quá mức.
đã ghi c:\Users\lamdu\Downloads\BanNH_QE210137_TRENDING_ANALYTICS-main (2)\BanNH_QE210137_TRENDING_ANALYTICS-main\data\processed\xgb_best_params.json


## 5. Chỉ số trên CV (điểm ngoài fold), đặt cạnh hai mô hình mốc

Đây là chỉ số CV trên tập train, **chưa phải tập test**. Accuracy luôn đọc cạnh mô hình mốc: luôn đoán 0 đã đạt khoảng 96%.

In [ ]:
oof_xgb = res["oof_score"]
rows = {"XGBoost (bộ C)": M.evaluate(y, oof_xgb, thr)}
for name, base in M.baselines().items():
    oof = np.zeros(len(y))
    for tr, va in M.cv_folds(y, groups):
        base.fit(X.iloc[tr], y[tr])
        oof[va] = base.predict_proba(X.iloc[va])[:, 1]
    rows[name] = M.evaluate(y, oof, 0.5)
cv_table = pd.DataFrame(rows).T
display(cv_table.round(4))

,accuracy,precision,recall,f1,pr_auc,roc_auc,precision_top1pct,precision_top5pct,tn,fp,fn,tp
XGBoost (bộ C),0.9399,0.2785,0.3492,0.3099,0.2119,0.8344,0.4024,0.2696,7552.0,285.0,205.0,110.0
mốc: luôn đoán 0,0.9614,0.0000,0.0000,0.0000,0.0386,0.5000,0.0366,0.0392,7837.0,0.0,315.0,0.0
mốc: ngẫu nhiên theo tỉ lệ,0.9216,0.0235,0.0254,0.0244,0.0383,0.4915,0.0244,0.0270,7505.0,332.0,307.0,8.0


## 6. Giao cho duc và nhận xét (điền sau khi chạy)

**Giao cho duc:** file `data/processed/xgb_best_params.json` (tham số, ngưỡng, bộ feature C cho cây). duc dùng đúng tham số này để so bộ C với bộ C bỏ thời gian (`drop_time=True`).

**Nhận xét (3–5 câu, điền số thật):**
1. PR-AUC CV của XGBoost = … so với tỉ lệ trending = … (cao gấp … lần mức ngẫu nhiên).
2. Accuracy = … nhưng mô hình luôn đoán 0 đã đạt …; vì vậy dựa vào recall = …, precision = …, F1 = ….
3. Độ lệch giữa các fold = … nên …
4. Giới hạn: dữ liệu quan sát một đợt crawl ngắn; mô hình chỉ cho thấy mức "đi kèm", không chứng minh giờ đăng gây ra trending; ngưỡng chọn trên điểm gộp của các fold.